# CMPE281: real CARLA on Google Colab
**Rishi Sheth · Group 8 · C2/C3 component demonstration**

Upload this notebook to your existing SJSU Google Colab account, select a GPU runtime,
and run the cells **one at a time**. Everything needed for the CARLA smoke test is
embedded here. No new online account, Drive mount, SSH, remote desktop or tunnel is used.

**Status at delivery: prepared, not executed on your Colab runtime.** The matching
Python 3.12/CARLA 0.9.16 client and audio-validation imports were checked locally.
Actual server, camera and movement claims require the evidence produced below.
Colab does not guarantee GPU availability or runtime duration. Save your evidence
before disconnecting. This is an interactive notebook computation, following the
[Colab FAQ](https://research.google.com/colaboratory/faq.html).

The official server archive is **8,346,095,504 bytes (8.35 GB / 7.77 GiB)**. Allow
**25 GiB free** for download, extraction and evidence. The first checks stop before
that download if the runtime cannot expose a real GPU through Vulkan. A T4 showing
up in `nvidia-smi` alone does not establish Vulkan rendering support.


In [ ]:
from pathlib import Path
import ctypes, ctypes.util, json, os, platform, pwd, shutil, signal, socket, subprocess, sys, time
from IPython.display import display, Image, HTML
ROOT = Path('/content/cmpe281')
EVIDENCE = ROOT / 'evidence'
CARLA_ROOT = ROOT / 'carla' / 'server'
CLIENT_PYTHON = ROOT / 'env' / 'bin' / 'python'
SMOKE_SCRIPT = ROOT / 'carla_smoke.py'
RUN_MODE = 'camera'  # Keep camera for F11. 'physics' is an explicitly limited -nullrhi fallback.
PREFLIGHT_OK = False
USER_PREFLIGHT_OK = False
SERVER_READY = False
for directory in (ROOT, EVIDENCE, CARLA_ROOT, ROOT / 'tools'):
    directory.mkdir(parents=True, exist_ok=True)

SOURCES = {'probe.py': '#!/usr/bin/env python3\n"""Inspect a prospective CARLA host without installing or downloading anything."""\nimport argparse\nimport ctypes as c\nfrom ctypes.util import find_library\nfrom datetime import datetime, timezone\nimport glob\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport struct\nimport subprocess\nimport urllib.request\n\n\ndef command(args):\n    executable = shutil.which(args[0])\n    if not executable:\n        return {"available": False}\n    try:\n        result = subprocess.run([executable, *args[1:]], capture_output=True, text=True, timeout=15)\n        return {"available": True, "exit_code": result.returncode,\n                "stdout": result.stdout[:12000], "stderr": result.stderr[:4000]}\n    except Exception as exc:\n        return {"available": True, "error": f"{type(exc).__name__}: {exc}"}\n\n\ndef vulkan_devices():\n    class Application(c.Structure):\n        _fields_ = [("sType", c.c_uint32), ("pNext", c.c_void_p), ("name", c.c_char_p), ("version", c.c_uint32),\n                    ("engine", c.c_char_p), ("engine_version", c.c_uint32), ("api", c.c_uint32)]\n    class Info(c.Structure):\n        _fields_ = [("sType", c.c_uint32), ("pNext", c.c_void_p), ("flags", c.c_uint32),\n                    ("app", c.POINTER(Application)), ("layers", c.c_uint32), ("layer_names", c.c_void_p),\n                    ("extensions", c.c_uint32), ("extension_names", c.c_void_p)]\n    library = find_library("vulkan")\n    if not library:\n        return {"loader": None, "error": "Vulkan loader missing", "devices": []}\n    v = c.CDLL(library)\n    v.vkCreateInstance.argtypes = [c.POINTER(Info), c.c_void_p, c.POINTER(c.c_void_p)]\n    v.vkCreateInstance.restype = c.c_int32\n    v.vkEnumeratePhysicalDevices.argtypes = [c.c_void_p, c.POINTER(c.c_uint32), c.c_void_p]\n    v.vkEnumeratePhysicalDevices.restype = c.c_int32\n    v.vkGetPhysicalDeviceProperties.argtypes = [c.c_void_p, c.c_void_p]\n    v.vkDestroyInstance.argtypes = [c.c_void_p, c.c_void_p]\n    app = Application(0, None, b"CARLA host preflight", 0, b"preflight", 0, 1 << 22)\n    info = Info(1, None, 0, c.pointer(app), 0, None, 0, None)\n    instance = c.c_void_p()\n    status = v.vkCreateInstance(c.byref(info), None, c.byref(instance))\n    result = {"loader": library, "create_instance_result": status, "devices": []}\n    if status != 0:\n        return result\n    try:\n        count = c.c_uint32()\n        v.vkEnumeratePhysicalDevices(instance, c.byref(count), None)\n        devices = (c.c_void_p * count.value)()\n        v.vkEnumeratePhysicalDevices(instance, c.byref(count), devices)\n        for device in devices:\n            properties = c.create_string_buffer(4096)\n            v.vkGetPhysicalDeviceProperties(device, properties)\n            api, driver, vendor, model, kind = struct.unpack_from("5I", properties.raw)\n            name = properties.raw[20:276].split(b"\\0", 1)[0].decode(errors="replace")\n            result["devices"].append({"name": name, "vendor_id": hex(vendor), "device_id": hex(model),\n                                      "device_type": kind, "api_version": f"{api >> 22}.{(api >> 12) & 1023}.{api & 4095}"})\n    finally:\n        v.vkDestroyInstance(instance, None)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--work-dir", type=Path, default=Path.cwd())\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--require-gpu", action="store_true", help="Fail unless Vulkan enumerates a real integrated/discrete GPU")\n    parser.add_argument("--api-url", help="Optionally check the SSH-forwarded API, normally http://127.0.0.1:18000")\n    args = parser.parse_args()\n    work = args.work_dir.resolve()\n    existing = work\n    while not existing.exists():\n        existing = existing.parent\n    result = {"observed_at": datetime.now(timezone.utc).isoformat(), "platform": platform.platform(),\n              "python": platform.python_version(), "work_dir": str(work), "uid": os.getuid(),\n              "free_disk_gib": round(shutil.disk_usage(existing).free / 1024 ** 3, 2),\n              "ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024 ** 3, 2),\n              "nvidia_smi": command(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"]),\n              "vulkan_icd_files": glob.glob("/usr/share/vulkan/icd.d/*.json") + glob.glob("/etc/vulkan/icd.d/*.json"),\n              "graphics_loader": {name: find_library(name) for name in ("vulkan", "GL", "EGL")}}\n    try:\n        result["vulkan"] = vulkan_devices()\n    except Exception as exc:\n        result["vulkan"] = {"error": f"{type(exc).__name__}: {exc}", "devices": []}\n    result["hardware_vulkan"] = any(device["device_type"] in (1, 2) for device in result["vulkan"]["devices"])\n    if args.api_url:\n        try:\n            with urllib.request.urlopen(args.api_url.rstrip("/") + "/health", timeout=5) as response:\n                result["api_health"] = {"status": response.status, "body": json.load(response)}\n        except Exception as exc:\n            result["api_health"] = {"error": f"{type(exc).__name__}: {exc}"}\n    encoded = json.dumps(result, indent=2) + "\\n"\n    if args.output:\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        args.output.write_text(encoded)\n    print(encoded, end="")\n    return 3 if args.require_gpu and not result["hardware_vulkan"] else 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'carla_smoke.py': '#!/usr/bin/env python3\n"""Smoke-test a real, already running CARLA 0.9.16 server.\n\nCreates one owned vehicle and, unless --no-camera, one owned RGB camera. The\nvehicle drives for half the duration then stops. World settings are never changed.\nOnly actors created by this process are destroyed on exit.\n"""\nimport argparse\nfrom datetime import datetime, timezone\nimport json\nimport math\nfrom pathlib import Path\nimport queue\nimport time\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef emit(stage, **values):\n    print(json.dumps({"stage": stage, **values}), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--host", default="127.0.0.1")\n    parser.add_argument("--port", type=int, default=2000)\n    parser.add_argument("--traffic-manager-port", type=int, default=8001)\n    parser.add_argument("--seconds", type=float, default=20)\n    parser.add_argument("--timeout", type=float, default=20)\n    parser.add_argument("--no-camera", action="store_true", help="For a real -nullrhi physics-only server")\n    parser.add_argument("--external-control", action="store_true", help="Keep the owned actor for an edge client; do not start/stop its autopilot here")\n    parser.add_argument("--output-dir", type=Path, default=Path(__file__).with_name("carla-smoke"))\n    args = parser.parse_args()\n    if args.seconds <= 0 or args.timeout <= 0:\n        parser.error("--seconds and --timeout must be positive")\n    import carla\n    from PIL import Image\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    evidence = {"started_at": now(), "status": "running",\n                "server_host": args.host, "server_port": args.port, "camera_requested": not args.no_camera,\n                "camera_frames": 0, "telemetry": [], "owned_actor_ids": [], "cleanup": []}\n    vehicle, sensor = None, None\n    frame_queue = queue.Queue(maxsize=1)\n    result = 1\n\n    def receive(image):\n        try:\n            frame_queue.put_nowait(image)\n        except queue.Full:\n            try:\n                frame_queue.get_nowait()\n            except queue.Empty:\n                pass\n            try:\n                frame_queue.put_nowait(image)\n            except queue.Full:\n                pass\n\n    def save_frame(image):\n        frame = Image.frombytes("RGBA", (image.width, image.height), bytes(image.raw_data), "raw", "BGRA").convert("RGB")\n        frame.save(args.output_dir / "latest-frame.jpg", format="JPEG", quality=85)\n        evidence["camera_frames"] += 1\n        evidence["last_camera_frame"] = image.frame\n\n    try:\n        client = carla.Client(args.host, args.port)\n        client.set_timeout(args.timeout)\n        evidence["client_version"] = client.get_client_version()\n        evidence["server_version"] = client.get_server_version()\n        world = client.get_world()\n        carla_map = world.get_map()\n        settings = world.get_settings()\n        evidence["town"] = carla_map.name\n        evidence["synchronous_mode"] = settings.synchronous_mode\n        evidence["no_rendering_mode"] = settings.no_rendering_mode\n        # Check a live world is advancing before creating any actors. In a\n        # synchronous world its existing clock owner must keep ticking it.\n        before = world.get_snapshot().frame\n        snapshot = world.wait_for_tick(args.timeout)\n        if snapshot is None or snapshot.frame <= before:\n            raise RuntimeError("CARLA world is not advancing; retain or start its existing clock owner")\n        emit("connected", server_version=evidence["server_version"], town=carla_map.name,\n             synchronous_mode=settings.synchronous_mode, frame=snapshot.frame)\n        library = world.get_blueprint_library()\n        candidates = [blueprint for blueprint in library.filter("vehicle.*")\n                      if blueprint.has_attribute("number_of_wheels")\n                      and blueprint.get_attribute("number_of_wheels").as_int() == 4]\n        if not candidates:\n            raise RuntimeError("This CARLA world has no four-wheel vehicle blueprints")\n        candidates.sort(key=lambda blueprint: (blueprint.id != "vehicle.tesla.model3", blueprint.id))\n        blueprint = candidates[0]\n        if blueprint.has_attribute("role_name"):\n            blueprint.set_attribute("role_name", "rishi-c2-c3-smoke")\n        for spawn in carla_map.get_spawn_points():\n            vehicle = world.try_spawn_actor(blueprint, spawn)\n            if vehicle is not None:\n                break\n        if vehicle is None:\n            raise RuntimeError("No unoccupied vehicle spawn point was available")\n        evidence["owned_actor_ids"].append(vehicle.id)\n        evidence["vehicle_id"] = vehicle.id\n        evidence["vehicle_type"] = vehicle.type_id\n        if not args.no_camera:\n            camera = library.find("sensor.camera.rgb")\n            for key, value in {"image_size_x": "320", "image_size_y": "180", "sensor_tick": "0.5", "fov": "90"}.items():\n                camera.set_attribute(key, value)\n            sensor = world.spawn_actor(camera, carla.Transform(carla.Location(x=1.5, z=2.2)), attach_to=vehicle)\n            evidence["owned_actor_ids"].append(sensor.id)\n            evidence["camera_id"] = sensor.id\n            sensor.listen(receive)\n        if not args.external_control:\n            client.get_trafficmanager(args.traffic_manager_port)\n            vehicle.set_autopilot(True, args.traffic_manager_port)\n        (args.output_dir / "owned-actors.json").write_text(json.dumps({\n            "vehicle_id": vehicle.id, "camera_id": sensor.id if sensor else None,\n            "server_host": args.host, "server_port": args.port, "external_control": args.external_control,\n        }, indent=2) + "\\n")\n        emit("actors_created", vehicle_id=vehicle.id, camera_id=sensor.id if sensor else None)\n        started, last_report, stopped = time.monotonic(), -1.0, False\n        while time.monotonic() - started < args.seconds:\n            snapshot = world.wait_for_tick(args.timeout)\n            elapsed = time.monotonic() - started\n            if not args.external_control and elapsed >= args.seconds / 2 and not stopped:\n                vehicle.set_autopilot(False, args.traffic_manager_port)\n                vehicle.apply_control(carla.VehicleControl(throttle=0, brake=1))\n                stopped = True\n                emit("route_stopped", after_seconds=round(elapsed, 2))\n            try:\n                save_frame(frame_queue.get_nowait())\n            except queue.Empty:\n                pass\n            if elapsed - last_report >= 1:\n                transform = vehicle.get_transform()\n                geo = carla_map.transform_to_geolocation(transform.location)\n                velocity = vehicle.get_velocity()\n                row = {"observed_at": now(), "frame": snapshot.frame, "simulation_seconds": snapshot.timestamp.elapsed_seconds,\n                       "lat": geo.latitude, "lon": geo.longitude,\n                       "speed_kmh": 3.6 * math.sqrt(velocity.x ** 2 + velocity.y ** 2 + velocity.z ** 2),\n                       "heading": (transform.rotation.yaw + 90) % 360, "route_stopped": stopped}\n                evidence["telemetry"].append(row)\n                emit("telemetry", **row, camera_frames=evidence["camera_frames"])\n                last_report = elapsed\n        try:\n            save_frame(frame_queue.get_nowait())\n        except queue.Empty:\n            pass\n        if len(evidence["telemetry"]) < 2:\n            raise RuntimeError("Insufficient live telemetry samples")\n        if not args.no_camera and evidence["camera_frames"] == 0:\n            raise RuntimeError("Real RGB camera returned no frames; check rendering and Vulkan")\n        evidence["status"] = "passed"\n        evidence["max_speed_kmh"] = max(row["speed_kmh"] for row in evidence["telemetry"])\n        evidence["motion_observed"] = evidence["max_speed_kmh"] > 0.1\n        if not evidence["motion_observed"]:\n            raise RuntimeError("The owned vehicle did not show motion; inspect Traffic Manager and spawn conditions")\n        result = 0\n    except KeyboardInterrupt:\n        evidence["status"] = "interrupted"\n    except Exception as exc:\n        evidence["status"] = "failed"\n        evidence["error"] = f"{type(exc).__name__}: {exc}"\n        emit("failed", error=evidence["error"])\n    finally:\n        if sensor is not None:\n            try:\n                sensor.stop()\n            except Exception as exc:\n                evidence["cleanup"].append({"actor_id": sensor.id, "stop_error": str(exc)})\n        for actor in (sensor, vehicle):\n            if actor is not None:\n                try:\n                    destroyed = actor.destroy()\n                    evidence["cleanup"].append({"actor_id": actor.id, "destroyed": destroyed})\n                    if not destroyed:\n                        result = 1\n                except Exception as exc:\n                    evidence["cleanup"].append({"actor_id": actor.id, "destroy_error": str(exc)})\n                    result = 1\n        evidence["finished_at"] = now()\n        if result != 0 and evidence["status"] == "passed":\n            evidence["status"] = "cleanup_failed"\n        path = args.output_dir / "smoke-evidence.json"\n        path.write_text(json.dumps(evidence, indent=2) + "\\n")\n        emit("finished", status=evidence["status"], evidence=str(path), camera_frames=evidence["camera_frames"])\n    return result\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'}
for filename, source in SOURCES.items():
    (ROOT / filename).write_text(source)
def run(args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)
print('Embedded helpers ready in', ROOT)
print('Kernel:', sys.version, '| Platform:', platform.platform())


## 1. GPU, memory, disk and actual Vulkan preflight
This cell may install the small `libvulkan1` loader in the disposable Colab VM.
It does **not** replace the NVIDIA kernel driver. If an existing NVIDIA graphics
library is present but its ICD descriptor is missing, it creates a workspace-local
descriptor for that actual library. A missing graphics driver remains a clear stop.


In [ ]:
PREFLIGHT_OK = False
USER_PREFLIGHT_OK = False
assert RUN_MODE in ('camera', 'physics')
print('Kernel Python:', sys.version)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
                     text=True, capture_output=True) if shutil.which('nvidia-smi') else None
print(gpu.stdout if gpu and gpu.returncode == 0 else 'No NVIDIA GPU visible to nvidia-smi.')
if RUN_MODE == 'camera' and (gpu is None or gpu.returncode != 0):
    raise RuntimeError('Select a Colab GPU runtime first. No CARLA archive downloaded.')
if not ctypes.util.find_library('vulkan') and os.geteuid() == 0:
    run(['apt-get', '-qq', 'update'])
    run(['apt-get', 'install', '-y', '-qq', 'libvulkan1'])
graphics_library = ctypes.util.find_library('GLX_nvidia')
import glob
existing_nvidia_icd = glob.glob('/usr/share/vulkan/icd.d/*nvidia*.json') + glob.glob('/etc/vulkan/icd.d/*nvidia*.json')
if graphics_library and not existing_nvidia_icd:
    try:
        driver = ctypes.CDLL(graphics_library)
        entry = driver.vk_icdGetInstanceProcAddr
        entry.argtypes = [ctypes.c_void_p, ctypes.c_char_p]
        entry.restype = ctypes.c_void_p
        encoded_version = ctypes.c_uint32(1 << 22)
        address = entry(None, b'vkEnumerateInstanceVersion')
        if address:
            query_version = ctypes.CFUNCTYPE(ctypes.c_int32, ctypes.POINTER(ctypes.c_uint32))(address)
            query_version(ctypes.byref(encoded_version))
        api = encoded_version.value
        api_version = f'{api >> 22}.{(api >> 12) & 1023}.{api & 4095}'
        icd = ROOT / 'nvidia_icd.json'
        icd.write_text(json.dumps({'file_format_version': '1.0.0', 'ICD': {
            'library_path': graphics_library, 'api_version': api_version}}))
        os.environ['VK_DRIVER_FILES'] = str(icd)
        print('Using existing NVIDIA graphics library:', graphics_library)
    except (OSError, AttributeError):
        pass
probe = subprocess.run([sys.executable, str(ROOT / 'probe.py'), '--work-dir', str(ROOT)],
                       text=True, capture_output=True)
print(probe.stdout)
if probe.stderr:
    print(probe.stderr[-4000:])
probe.check_returncode()
host = json.loads(probe.stdout)
(EVIDENCE / 'host-preflight.json').write_text(json.dumps(host, indent=2))
if platform.machine() != 'x86_64' or platform.system() != 'Linux':
    raise RuntimeError('This package requires Linux x86-64. No archive downloaded.')
if host['free_disk_gib'] < 25:
    raise RuntimeError('Need at least 25 GiB free. No archive downloaded.')
if host['ram_gib'] < 8:
    raise RuntimeError('This runtime has less than the notebook\'s 8 GiB working-memory budget. No archive downloaded.')
if RUN_MODE == 'camera' and not host['hardware_vulkan']:
    raise RuntimeError('GPU/CUDA may be visible, but Vulkan did not enumerate a hardware GPU. '
                       'This runtime is missing usable graphics-driver/ICD support. No 8 GB archive downloaded. '
                       'Keep this output for diagnosis; a T4 name alone is insufficient.')
PREFLIGHT_OK = True
print('PREFLIGHT PASSED:', RUN_MODE, '| no CARLA archive has been downloaded yet.')


## 2. Isolated matching client and non-root simulator user
The notebook kernel stays unchanged. This installs Python 3.12, the official
`carla==0.9.16` client and Pillow in `/content/cmpe281/env`. Unreal runs under an
ephemeral **Unix user inside this VM**, because the Colab kernel commonly runs as
root. This is not a new Google or cloud account. The next probe verifies that this
non-root user can also see the actual Vulkan GPU before the large archive download.


In [ ]:
USER_PREFLIGHT_OK = False
assert PREFLIGHT_OK, 'Run and pass preflight first.'
tools = ROOT / 'tools'
mamba = tools / 'bin' / 'micromamba'
if not CLIENT_PYTHON.exists():
    if not mamba.exists():
        archive = tools / 'micromamba.tar.bz2'
        run(['curl', '-fL', '--retry', '3', 'https://micro.mamba.pm/api/micromamba/linux-64/latest', '-o', archive])
        import tarfile
        with tarfile.open(archive, 'r:bz2') as tar:
            member = tar.getmember('bin/micromamba')
            mamba.parent.mkdir(parents=True, exist_ok=True)
            with tar.extractfile(member) as source, mamba.open('wb') as target:
                shutil.copyfileobj(source, target)
        mamba.chmod(0o755)
    run([mamba, 'create', '-y', '--root-prefix', ROOT / 'mamba', '--prefix', ROOT / 'env',
         '-c', 'conda-forge', 'python=3.12', 'pip'])
run([CLIENT_PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check',
     '--cache-dir', ROOT / 'pip-cache', 'carla==0.9.16', 'Pillow>=10'])
run([CLIENT_PYTHON, '-m', 'pip', 'check'])
run([CLIENT_PYTHON, '-c', "import carla,sys; print(sys.version); print('CARLA client:',carla.Client('127.0.0.1',2000).get_client_version())"])
SERVER_USER = 'cmpe281-carla'
RUN_AS = []
if os.geteuid() == 0:
    try:
        account = pwd.getpwnam(SERVER_USER)
    except KeyError:
        run(['useradd', '--system', '--user-group', '--create-home', '--home-dir', ROOT / 'carla-user',
             '--shell', '/bin/bash', SERVER_USER])
        account = pwd.getpwnam(SERVER_USER)
    import grp
    groups = [name for name in ('video', 'render') if any(g.gr_name == name for g in grp.getgrall())]
    if groups:
        run(['usermod', '-a', '-G', ','.join(groups), SERVER_USER])
    for directory in (ROOT / 'carla', CARLA_ROOT, ROOT / 'carla-user'):
        directory.mkdir(parents=True, exist_ok=True)
        os.chown(directory, account.pw_uid, account.pw_gid)
    RUN_AS = ['runuser', '-u', SERVER_USER, '--']
nonroot = subprocess.run([*RUN_AS, str(CLIENT_PYTHON), str(ROOT / 'probe.py'), '--work-dir', str(ROOT)],
                         text=True, capture_output=True)
print(nonroot.stdout)
if nonroot.stderr:
    print(nonroot.stderr[-4000:])
nonroot.check_returncode()
user_host = json.loads(nonroot.stdout)
(EVIDENCE / 'server-user-preflight.json').write_text(json.dumps(user_host, indent=2))
if user_host['uid'] == 0:
    raise RuntimeError('Unreal server user is still root; stop before the archive download.')
if RUN_MODE == 'camera' and not user_host['hardware_vulkan']:
    raise RuntimeError('The non-root server user cannot access a Vulkan GPU. No CARLA archive downloaded.')
USER_PREFLIGHT_OK = True
print('Matching client and non-root', RUN_MODE, 'preflight passed.')


## 3. Download the official server on Colab, then extract
This is the **large cloud download**. Do it only after both preflights passed.
It is resumable and uses the official
[CARLA 0.9.16 release](https://github.com/carla-simulator/carla/releases/tag/0.9.16).
The additional maps download is unnecessary. Archive size and gzip extraction are
checked; the release does not supply a checksum in this notebook.


In [ ]:
assert PREFLIGHT_OK and USER_PREFLIGHT_OK, 'Both preflight stages must pass before the large download.'
archive = ROOT / 'carla' / 'CARLA_0.9.16.tar.gz'
expected_bytes = 8346095504
url = 'https://downloads.carlasim.com/Linux/CARLA_0.9.16.tar.gz'
started = time.monotonic()
if not archive.exists() or archive.stat().st_size != expected_bytes:
    run(['curl', '--fail', '--location', '--continue-at', '-', '--retry', '3', '--connect-timeout', '20',
         '--output', archive, url])
assert archive.stat().st_size == expected_bytes, 'Incomplete or unexpected archive size.'
complete = CARLA_ROOT / '.extraction-complete'
if not complete.exists():
    run([*RUN_AS, 'tar', '-xzf', archive, '-C', CARLA_ROOT])
    assert (CARLA_ROOT / 'CarlaUE4.sh').is_file(), 'CARLA launcher missing.'
    complete.write_text('CARLA 0.9.16 archive extracted successfully.\n')
print('Server ready on disk:', CARLA_ROOT, '| preparation seconds:', round(time.monotonic() - started, 1))


## 4. Start the actual server and load a small town
Camera mode uses official low-quality offscreen rendering, which still produces
real RGB camera data. Physics fallback uses `-nullrhi` and has no camera evidence.
See [rendering modes](https://carla.readthedocs.io/en/0.9.16/adv_rendering_options/)
and the [GPU-free primary server](https://carla.readthedocs.io/en/0.9.16/adv_multigpu/).
This cell starts only a local notebook-owned process; it opens no external service.


In [ ]:
SERVER_READY = False
assert USER_PREFLIGHT_OK and (CARLA_ROOT / '.extraction-complete').exists()
def rpc_open():
    try:
        with socket.create_connection(('127.0.0.1', 2000), timeout=1):
            return True
    except OSError:
        return False
if rpc_open():
    raise RuntimeError('Port 2000 already has a server. Use the cleanup cell for a notebook-owned server first.')
server_log = EVIDENCE / 'carla-server.log'
flags = (['-RenderOffScreen', '-quality-level=Low', '-nosound', '-ResX=640', '-ResY=360']
         if RUN_MODE == 'camera' else ['-nullrhi', '-nosound'])
server_env = dict(os.environ)
server_env['XDG_CONFIG_HOME'] = str(ROOT / 'carla-user' / 'config')
server_env['XDG_CACHE_HOME'] = str(ROOT / 'carla-user' / 'cache')
server_env['CARLA_CACHE_DIR'] = str(ROOT / 'carla-user' / 'carla-cache')
with server_log.open('ab', buffering=0) as log:
    CARLA_PROC = subprocess.Popen([*RUN_AS, str(CARLA_ROOT / 'CarlaUE4.sh'), *flags,
                                  '-carla-rpc-port=2000', '-carla-streaming-port=2001'],
                                 cwd=CARLA_ROOT, env=server_env, stdin=subprocess.DEVNULL,
                                 stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
start_ticks = Path(f'/proc/{CARLA_PROC.pid}/stat').read_text().rpartition(')')[2].split()[19]
(ROOT / 'carla-session.json').write_text(json.dumps({'pid': CARLA_PROC.pid, 'start_ticks': start_ticks,
    'root': str(CARLA_ROOT), 'run_mode': RUN_MODE, 'owner': 'this notebook'}, indent=2))
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    if CARLA_PROC.poll() is not None:
        print(server_log.read_text(errors='replace')[-12000:])
        raise RuntimeError('Actual CARLA server exited; retain the log. No successful run is claimed.')
    if rpc_open():
        break
    time.sleep(2)
else:
    print(server_log.read_text(errors='replace')[-12000:])
    raise RuntimeError('CARLA RPC did not become ready within 180 seconds; inspect the actual server log.')
load_town = "import carla,json\nc=carla.Client('127.0.0.1',2000); c.set_timeout(120)\nmaps=c.get_available_maps()\ntown=next((name for name in maps if name.endswith('/Town02')),None)\nworld=c.load_world(town) if town else c.get_world()\nprint(json.dumps({'server_version':c.get_server_version(),'map':world.get_map().name,'frame':world.get_snapshot().frame}))\n"
proof = subprocess.run([str(CLIENT_PYTHON), '-c', load_town], text=True, capture_output=True)
print(proof.stdout)
if proof.stderr:
    print(proof.stderr[-4000:])
proof.check_returncode()
(EVIDENCE / 'server-roundtrip.json').write_text(proof.stdout)
SERVER_READY = True
print('Real CARLA server responded to the matching client.')


## 5. Real actor, camera, movement and stop smoke test
This spawns an actual vehicle and a small attached camera, observes real telemetry,
stops the vehicle halfway through, and destroys only its owned actors afterward.
It changes no existing world settings. A successful camera smoke needs received
camera frames **and observed vehicle motion**. Errors are retained as failed
evidence; a startup or rendering failure is not converted into a mock success.


In [ ]:
assert SERVER_READY, 'Run the actual server round-trip first.'
smoke_dir = EVIDENCE / 'smoke'
command = [str(CLIENT_PYTHON), str(SMOKE_SCRIPT), '--seconds', '20', '--output-dir', str(smoke_dir)]
if RUN_MODE == 'physics':
    command.append('--no-camera')
smoke = subprocess.run(command, text=True)
proof_path = smoke_dir / 'smoke-evidence.json'
if proof_path.exists():
    smoke_result = json.loads(proof_path.read_text())
    print(json.dumps({key: value for key, value in smoke_result.items() if key != 'telemetry'}, indent=2))
    import html
    rows = smoke_result.get('telemetry', [])
    columns = ['frame', 'lat', 'lon', 'speed_kmh', 'heading', 'route_stopped']
    table = '<table><thead><tr>' + ''.join('<th>'+name+'</th>' for name in columns) + '</tr></thead><tbody>'
    for row in rows:
        table += '<tr>' + ''.join('<td>'+html.escape(str(row.get(name)))+'</td>' for name in columns) + '</tr>'
    display(HTML(table + '</tbody></table>'))
frame = smoke_dir / 'latest-frame.jpg'
if frame.exists():
    display(Image(filename=str(frame), width=640))
smoke.check_returncode()
print('Actual CARLA', RUN_MODE, 'smoke passed. The JSON and JPEG above are produced by this run.')


## 6. Download evidence before the runtime ends
The ZIP contains actual probes, server logs, received image and smoke JSON. It does
not include the multi-gigabyte server archive. No new sound dataset or model
training is needed for this server/camera test. The existing C2/C3 repository has
separately attributed recorded scenarios for the component upload flow.


In [ ]:
import zipfile
evidence_zip = ROOT / 'Rishi_Sheth_CARLA_Colab_Evidence.zip'
with zipfile.ZipFile(evidence_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EVIDENCE.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(EVIDENCE))
print('Evidence:', evidence_zip, '| bytes:', evidence_zip.stat().st_size)
from google.colab import files
files.download(str(evidence_zip))


## 7. Stop only this notebook's CARLA server
Run this when finished, then disconnect/delete the Colab runtime. It checks the
recorded process identity before signaling its own process group. Other processes
and data are preserved. Save your evidence first.


In [ ]:
state_file = ROOT / 'carla-session.json'
if state_file.exists():
    state = json.loads(state_file.read_text())
    pid = state['pid']
    proc = Path(f'/proc/{pid}')
    if proc.exists():
        ticks = (proc / 'stat').read_text().rpartition(')')[2].split()[19]
        command = (proc / 'cmdline').read_bytes().replace(b'\0', b' ').decode(errors='replace')
        if ticks != state['start_ticks'] or str(CARLA_ROOT) not in command or os.getpgid(pid) != pid:
            raise RuntimeError('Recorded PID no longer identifies this notebook-owned server; no signal sent.')
        os.killpg(pid, signal.SIGTERM)
        print('Sent SIGTERM only to this notebook-owned CARLA process group:', pid)
    else:
        print('The recorded server has already exited.')
    state_file.unlink(missing_ok=True)
else:
    print('No notebook-owned server recorded.')
SERVER_READY = False


## Sources and execution boundary
- [Official CARLA 0.9.16 release](https://github.com/carla-simulator/carla/releases/tag/0.9.16)
- [CARLA package requirements](https://carla.readthedocs.io/en/0.9.16/start_quickstart/)
- [Rendering and offscreen camera modes](https://carla.readthedocs.io/en/0.9.16/adv_rendering_options/)
- [GPU-free `-nullrhi` primary server](https://carla.readthedocs.io/en/0.9.16/adv_multigpu/)
- [Official matching Python client](https://pypi.org/project/carla/0.9.16/)
- [Micromamba manual installation](https://mamba.readthedocs.io/en/latest/installation/micromamba-installation.html)
- [Colab resource availability and usage restrictions](https://research.google.com/colaboratory/faq.html)

The component UI/backend and full Docker stack are separate evidence. This notebook
establishes actual CARLA execution only when its real run succeeds. An SJSU account
does not establish a guaranteed GPU or a particular institutional paid entitlement.
